# LeetCode 1887 — Reduction Operations to Make the Array Elements Equal bằng Python

Link đề: https://leetcode.com/problems/reduction-operations-to-make-the-array-elements-equal/ · Độ khó: **Medium** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def reductionOperations(self, nums: List[int]) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng đúng từng thao tác** như đề mô tả (tìm max, tìm "lớn kế tiếp", hạ xuống) — chậm, dùng để kiểm chứng.
2. **Mô phỏng theo nhóm giá trị** bằng `Counter`: mỗi "tầng" giá trị hạ cả nhóm một lần.
3. **Đếm theo hạng (rank)**: mỗi phần tử cần đúng `rank(v)` thao tác — số giá trị **phân biệt nhỏ hơn** nó. Sắp tăng rồi cộng dồn (`O(n log n)`).
4. **Sắp giảm và cộng chỉ số tại mỗi điểm đổi giá trị** — góc nhìn "tầng trên đổ xuống".
5. **Counting sort** trên miền giá trị `≤ 5·10⁴` — `O(n + V)`.

## 1. Mục tiêu học

1. Đọc hiểu một thao tác "mô phỏng" và tìm **đại lượng bất biến** thay vì mô phỏng: mỗi phần tử đi xuống qua **từng giá trị phân biệt** một.
2. Định nghĩa **hạng (rank)** của giá trị trong tập giá trị phân biệt và chứng minh tổng số thao tác = `Σ rank(nums[i])`.
3. Thấy rằng quy tắc "chọn chỉ số nhỏ nhất khi hòa" **không ảnh hưởng** tới tổng số thao tác.
4. Cài lời giải bằng sắp xếp (hai chiều) và bằng đếm (counting) khi miền giá trị nhỏ.
5. Kiểm chứng bằng mô phỏng trực tiếp trên input ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Mục tiêu: làm mọi phần tử **bằng nhau**. Một thao tác gồm:
1. Tìm giá trị **lớn nhất** `largest`, chỉ số `i` của nó (nếu nhiều, chọn `i` nhỏ nhất).
2. Tìm giá trị **lớn kế tiếp** `nextLargest` — giá trị lớn nhất **nhỏ hơn nghiêm ngặt** `largest`.
3. Hạ `nums[i]` xuống `nextLargest`.

Trả về số thao tác để mọi phần tử bằng nhau.

**Ví dụ 1:** `nums = [5,1,3]` → `3`: `[3,1,3] → [1,1,3] → [1,1,1]`.

**Ví dụ 2:** `nums = [1,1,1]` → `0`.

**Ví dụ 3:** `nums = [1,1,2,2,3]` → `4`: `[1,1,2,2,2] → [1,1,1,2,2] → [1,1,1,1,2] → [1,1,1,1,1]`.

**Ràng buộc:** `1 ≤ nums.length ≤ 5·10⁴`, `1 ≤ nums[i] ≤ 5·10⁴`.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [([5, 1, 3], 3), ([1, 1, 1], 0), ([1, 1, 2, 2, 3], 4)]
for nums, want in examples:
    print(f"nums={nums} -> kỳ vọng {want}")

nums=[5, 1, 3] -> kỳ vọng 3
nums=[1, 1, 1] -> kỳ vọng 0
nums=[1, 1, 2, 2, 3] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Tập giá trị phân biệt và hạng

`sorted(set(nums))` cho các giá trị phân biệt tăng dần. Hạng của `v` = vị trí của `v` trong danh sách đó (giá trị nhỏ nhất có hạng 0). Dùng dict `{v: rank}`.

In [2]:
nums = [5, 1, 3, 3, 8]
vals = sorted(set(nums))
rank = {v: r for r, v in enumerate(vals)}
print("giá trị phân biệt:", vals, "| hạng:", rank)

giá trị phân biệt: [1, 3, 5, 8] | hạng: {1: 0, 3: 1, 5: 2, 8: 3}


### 3.2 `Counter` và duyệt theo khóa đã sắp

In [3]:
c = Counter([1, 1, 2, 2, 3])
for v in sorted(c, reverse=True):
    print(v, "xuất hiện", c[v], "lần")

3 xuất hiện 1 lần
2 xuất hiện 2 lần
1 xuất hiện 2 lần


### 3.3 Sắp xếp và phát hiện "điểm đổi giá trị"

Sau khi sắp, `a[i] != a[i − 1]` đánh dấu nơi bắt đầu một giá trị mới. Đếm số điểm đổi giá trị ⇒ số giá trị phân biệt.

In [4]:
a = sorted([5, 1, 3, 3, 8])
changes = [i for i in range(1, len(a)) if a[i] != a[i - 1]]
print(a, "điểm đổi giá trị tại chỉ số", changes)

[1, 3, 3, 5, 8] điểm đổi giá trị tại chỉ số [1, 3, 4]


### 3.4 Đếm theo mảng (counting) khi giá trị nhỏ

`cnt = [0] * (V + 1)`, `cnt[v] += 1`. Duyệt `v` tăng dần là duyệt mảng đã sắp, không cần `sort`.

### 3.5 Nén tọa độ (coordinate compression) bằng `bisect`

Một cách khác để lấy hạng: với `vals = sorted(set(nums))`, hạng của `v` chính là `bisect_left(vals, v)` (số giá trị phân biệt nhỏ hơn `v`). Cách này hữu ích khi cần hạng của giá trị **không có** trong mảng.

In [5]:
from bisect import bisect_left
vals = sorted(set([5, 1, 3, 3, 8]))
print([(v, bisect_left(vals, v)) for v in [1, 3, 5, 8, 4]], "(4 không có trong mảng: hạng 'chèn' = 2)")

[(1, 0), (3, 1), (5, 2), (8, 3), (4, 2)] (4 không có trong mảng: hạng 'chèn' = 2)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3 `[1,1,2,2,3]`: giá trị phân biệt `1 < 2 < 3`.
- Phần tử `3` phải đi `3 → 2 → 1`: **2** thao tác (hạng 2).
- Mỗi phần tử `2` phải đi `2 → 1`: **1** thao tác (hạng 1) — có hai phần tử ⇒ 2.
- Phần tử `1`: 0.

Tổng `2 + 2 = 4`. ✓

**Vì sao mỗi lần chỉ xuống đúng một "bậc"?** `nextLargest` là giá trị phân biệt **liền dưới** `largest` đang có trong mảng. Các giá trị ban đầu không bao giờ bị "bỏ qua" vì phần tử mang giá trị đó vẫn còn trong mảng (chỉ phần tử lớn nhất bị hạ). Vậy phần tử ban đầu `v` sẽ đi qua **mọi** giá trị phân biệt nhỏ hơn nó ⇒ `rank(v)` thao tác.

In [6]:
nums = [1, 1, 2, 2, 3]
vals = sorted(set(nums)); rank = {v: r for r, v in enumerate(vals)}
print([(v, rank[v]) for v in nums], "-> tổng hạng =", sum(rank[v] for v in nums))

[(1, 0), (1, 0), (2, 1), (2, 1), (3, 2)] -> tổng hạng = 4


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng từng thao tác

**Ý tưởng:** lặp đúng 3 bước của đề cho tới khi mọi phần tử bằng nhau. Mỗi thao tác quét mảng `O(n)`.

In [7]:
def reduction_simulate(nums):
    a = list(nums)
    ops = 0
    while True:
        largest = max(a)
        smaller = [v for v in a if v < largest]
        if not smaller:
            return ops
        nxt = max(smaller)
        i = a.index(largest)          # chỉ số nhỏ nhất có giá trị largest
        a[i] = nxt
        ops += 1

for nums, want in examples:
    print(reduction_simulate(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp:** số thao tác = đáp án, có thể tới `n · (D − 1)` với `D` số giá trị phân biệt (ví dụ mảng `1, 2, …, n` cho `n(n−1)/2` thao tác); mỗi thao tác `O(n)` ⇒ `O(n³)` worst. Space `O(n)`.

**Nút thắt:** mỗi thao tác chỉ thay đổi **một** phần tử, nhưng ta quét lại cả mảng.

In [8]:
for n in [60, 120, 200]:
    nums = list(range(1, n + 1))
    t0 = time.perf_counter(); r = reduction_simulate(nums)
    print(f"n={n} (1..n): {r} thao tác, {time.perf_counter() - t0:.3f}s")

n=60 (1..n): 1770 thao tác, 0.006s
n=120 (1..n): 7140 thao tác, 0.048s


n=200 (1..n): 19900 thao tác, 0.150s


### 5.1 Cách 1b — Mô phỏng theo nhóm giá trị

Thay vì hạ từng phần tử, hạ **cả nhóm** giá trị lớn nhất cùng lúc: nhóm có `c` phần tử ở giá trị `largest` tốn `c` thao tác để xuống `nextLargest`, rồi gộp vào nhóm đó.

In [9]:
def reduction_groups(nums):
    c = Counter(nums)
    vals = sorted(c, reverse=True)
    ops = 0
    for k in range(len(vals) - 1):
        top, nxt = vals[k], vals[k + 1]
        ops += c[top]                 # mỗi phần tử ở tầng top tốn 1 thao tác
        c[nxt] += c[top]              # gộp xuống tầng dưới
        c[top] = 0
    return ops

for nums, want in examples:
    print(reduction_groups(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


Trace mô phỏng theo nhóm trên `[1,1,2,2,3,3,3]` — mỗi bước "đổ" cả tầng trên cùng xuống tầng kế:

In [10]:
def trace_groups(nums):
    c = Counter(nums); vals = sorted(c, reverse=True); ops = 0
    print("các tầng (giá trị: số phần tử):", {v: c[v] for v in vals})
    for k in range(len(vals) - 1):
        top, nxt = vals[k], vals[k + 1]
        ops += c[top]
        print(f"đổ tầng {top} ({c[top]} phần tử) xuống {nxt}: +{c[top]} thao tác -> tầng {nxt} có {c[nxt] + c[top]}, tổng {ops}")
        c[nxt] += c[top]; c[top] = 0
    return ops

trace_groups([1, 1, 2, 2, 3, 3, 3])

các tầng (giá trị: số phần tử): {3: 3, 2: 2, 1: 2}
đổ tầng 3 (3 phần tử) xuống 2: +3 thao tác -> tầng 2 có 5, tổng 3
đổ tầng 2 (5 phần tử) xuống 1: +5 thao tác -> tầng 1 có 7, tổng 8


8

**Độ phức tạp:** `O(n + D log D)`. Đây đã là lời giải tốt, và nó gợi ý công thức đóng ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `v₀ < v₁ < … < v_{D−1}` là các giá trị phân biệt. Phần tử có giá trị ban đầu `v_r` tham gia đúng `r` thao tác (một lần ở mỗi bậc `v_r → v_{r−1} → … → v₀`). Do đó

`đáp án = Σ_i rank(nums[i])`.

**Cách tính 1 (sắp tăng):** duyệt `a` đã sắp tăng, giữ `r` = hạng hiện tại; mỗi khi `a[i] != a[i−1]` thì `r += 1`; cộng `r` vào đáp án.

**Cách tính 2 (sắp giảm):** duyệt `a` đã sắp giảm; tại mỗi điểm đổi giá trị `i` (`a[i] != a[i−1]`), tất cả `i` phần tử phía trước (lớn hơn) đều phải xuống thêm một bậc ⇒ cộng `i`. Đây chính là mô phỏng theo nhóm (Cách 1b) viết gọn.

**Invariant (sắp giảm):** sau khi xử lý tới chỉ số `i`, đáp án tạm = số thao tác cần để đưa mọi phần tử `a[0..i−1]` xuống giá trị `a[i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp tăng, cộng hạng

In [11]:
def reduction_rank_asc(nums):
    a = sorted(nums)
    ans = 0
    r = 0
    for i in range(1, len(a)):
        if a[i] != a[i - 1]:
            r += 1
        ans += r
    return ans

for nums, want in examples:
    print(reduction_rank_asc(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### 7.2 Cách 3 — Sắp giảm, cộng chỉ số tại điểm đổi giá trị

In [12]:
def reduction_desc(nums):
    a = sorted(nums, reverse=True)
    ans = 0
    for i in range(1, len(a)):
        if a[i] != a[i - 1]:
            ans += i              # i phần tử lớn hơn đều xuống thêm một bậc
    return ans

for nums, want in examples:
    print(reduction_desc(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### 7.3 Cách 4 — Counting sort `O(n + V)`

Từng đoạn:
1. `cnt[v]` cho `v ∈ [1, 5·10⁴]`.
2. Duyệt `v` tăng dần, giữ `r` = số giá trị phân biệt đã gặp **trước** `v` (tức hạng của `v`).
3. `ans += cnt[v] · r` rồi `r += 1` nếu `cnt[v] > 0`.

In [13]:
def reduction_counting(nums, V=5 * 10**4):
    cnt = [0] * (V + 1)
    for x in nums:
        cnt[x] += 1
    ans = 0
    r = 0
    for v in range(1, V + 1):
        c = cnt[v]
        if c:
            ans += c * r
            r += 1
    return ans

for nums, want in examples:
    print(reduction_counting(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp:** Time `O(n + V)` với `V = 5·10⁴`, Space `O(V)`.

### 7.4 Cách 5 — Một dòng với `Counter` + `enumerate`

`Σ_{r} r · count(v_r)` trên các giá trị phân biệt sắp tăng.

In [14]:
def reduction_oneliner(nums):
    c = Counter(nums)
    return sum(r * c[v] for r, v in enumerate(sorted(c)))

for nums, want in examples:
    print(reduction_oneliner(nums), "kỳ vọng", want)

3 kỳ vọng 3
0 kỳ vọng 0
4 kỳ vọng 4


**Độ phức tạp:** `O(n + D log D)`.

## 8. TRACE từng bước

Trace Cách 1 (mô phỏng) trên ví dụ 1 và ví dụ 3 — in mảng sau mỗi thao tác:

In [15]:
def trace_simulate(nums):
    a = list(nums); ops = 0
    print(f"bắt đầu: {a}")
    while True:
        largest = max(a); smaller = [v for v in a if v < largest]
        if not smaller:
            print(f"-> mọi phần tử bằng {largest}, tổng {ops} thao tác"); return ops
        nxt = max(smaller); i = a.index(largest); a[i] = nxt; ops += 1
        print(f"thao tác {ops}: largest={largest} tại i={i}, nextLargest={nxt} -> {a}")

trace_simulate([5, 1, 3])
trace_simulate([1, 1, 2, 2, 3])

bắt đầu: [5, 1, 3]
thao tác 1: largest=5 tại i=0, nextLargest=3 -> [3, 1, 3]
thao tác 2: largest=3 tại i=0, nextLargest=1 -> [1, 1, 3]
thao tác 3: largest=3 tại i=2, nextLargest=1 -> [1, 1, 1]
-> mọi phần tử bằng 1, tổng 3 thao tác
bắt đầu: [1, 1, 2, 2, 3]
thao tác 1: largest=3 tại i=4, nextLargest=2 -> [1, 1, 2, 2, 2]
thao tác 2: largest=2 tại i=2, nextLargest=1 -> [1, 1, 1, 2, 2]
thao tác 3: largest=2 tại i=3, nextLargest=1 -> [1, 1, 1, 1, 2]
thao tác 4: largest=2 tại i=4, nextLargest=1 -> [1, 1, 1, 1, 1]
-> mọi phần tử bằng 1, tổng 4 thao tác


4

Trace Cách 3 (sắp giảm) trên `[7, 7, 5, 5, 5, 2]`:

| i | a[i] | đổi giá trị? | cộng | ans |
|---|---|---|---|---|
| 1 | 7 | không | — | 0 |
| 2 | 5 | có | 2 (hai số 7 xuống 5) | 2 |
| 3 | 5 | không | — | 2 |
| 4 | 5 | không | — | 2 |
| 5 | 2 | có | 5 (năm số đầu xuống 2) | **7** |

In [16]:
def trace_desc(nums):
    a = sorted(nums, reverse=True); ans = 0
    print("sắp giảm:", a)
    for i in range(1, len(a)):
        if a[i] != a[i - 1]:
            ans += i
            print(f"i={i}: {a[i - 1]} -> {a[i]} đổi giá trị, {i} phần tử phía trước xuống thêm 1 bậc, ans={ans}")
    return ans

trace_desc([7, 7, 5, 5, 5, 2])
print("mô phỏng trực tiếp:", reduction_simulate([7, 7, 5, 5, 5, 2]))

sắp giảm: [7, 7, 5, 5, 5, 2]
i=2: 7 -> 5 đổi giá trị, 2 phần tử phía trước xuống thêm 1 bậc, ans=2
i=5: 5 -> 2 đổi giá trị, 5 phần tử phía trước xuống thêm 1 bậc, ans=7
mô phỏng trực tiếp: 7


In [17]:
def trace_rank(nums):
    a = sorted(nums); r = 0; ans = 0
    for i, v in enumerate(a):
        if i > 0 and v != a[i - 1]:
            r += 1
        ans += r
        print(f"a[{i}]={v}: hạng {r} -> cần {r} thao tác, tổng {ans}")
    return ans

trace_rank([5, 1, 3, 3])

a[0]=1: hạng 0 -> cần 0 thao tác, tổng 0
a[1]=3: hạng 1 -> cần 1 thao tác, tổng 1
a[2]=3: hạng 1 -> cần 1 thao tác, tổng 2
a[3]=5: hạng 2 -> cần 2 thao tác, tổng 4


4

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def reductionOperations(self, nums: List[int]) -> int:
        a = sorted(nums)
        ans = 0
        r = 0
        for i in range(1, len(a)):
            if a[i] != a[i - 1]:
                r += 1
            ans += r
        return ans

sol = Solution()
for nums, want in examples:
    got = sol.reductionOperations(nums)
    print(got, "OK" if got == want else "SAI")

3 OK
0 OK
4 OK


## 10. Vì sao thuật toán đúng

Gọi `v₀ < … < v_{D−1}` là các giá trị phân biệt **ban đầu**.

**Bổ đề 1 (tập giá trị chỉ co lại).** Mỗi thao tác đổi một phần tử từ `largest` thành `nextLargest` — một giá trị đã có trong mảng. Không giá trị mới nào xuất hiện ⇒ tập giá trị luôn là tập con của `{v₀, …, v_{D−1}}`.

**Bổ đề 2 (đi từng bậc).** Giả sử max hiện tại là `v_r` (`r ≥ 1`). Một phần tử chỉ bị hạ khi nó là max; mọi phần tử có giá trị **ban đầu** `v_{r−1}` chưa từng là max (vì lúc nào cũng còn phần tử `≥ v_r > v_{r−1}`), nên chúng vẫn giữ giá trị `v_{r−1}`. Kết hợp Bổ đề 1 (không có giá trị nào nằm giữa `v_{r−1}` và `v_r`), ta có `nextLargest = v_{r−1}`: phần tử bị hạ đi xuống **đúng một bậc**.

Do đó mỗi phần tử ban đầu `v_r` trải qua đúng `r` thao tác, và quá trình dừng khi mọi phần tử bằng `v₀` (giá trị nhỏ nhất không bao giờ bị hạ và không có gì nhỏ hơn). Tổng = `Σ rank`.

Quy tắc "chỉ số nhỏ nhất khi hòa" chỉ quyết định **thứ tự** các thao tác, không ảnh hưởng số thao tác của mỗi phần tử.

## 11. Phân tích Time / Space

Gọi `n ≤ 5·10⁴`, `D` số giá trị phân biệt, `V = 5·10⁴`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng thao tác | `O(ans · n)` (tới `O(n³)`) | `O(n)` | kiểm chứng |
| 1b. Mô phỏng theo nhóm | `O(n + D log D)` | `O(D)` | gợi ý công thức |
| 2. Sắp tăng + hạng | `O(n log n)` | `O(n)` | **nộp bài** |
| 3. Sắp giảm + chỉ số | `O(n log n)` | `O(n)` | góc nhìn "đổ xuống" |
| 4. Counting | `O(n + V)` | `O(V)` | giá trị nhỏ |
| 5. Counter + enumerate | `O(n + D log D)` | `O(D)` | gọn |

Đáp án tối đa `≈ n²/2 = 1.25·10⁹` (mảng toàn khác nhau) — vừa `int32` có dấu nhưng nên dùng `long` trong C++/Java.

In [19]:
random.seed(1)
nums_rand = [random.randint(1, 5 * 10**4) for _ in range(5 * 10**4)]
nums_distinct = list(range(1, 5 * 10**4 + 1))
for name, fn in [("groups", reduction_groups), ("asc", reduction_rank_asc), ("desc", reduction_desc),
                 ("count", reduction_counting), ("oneliner", reduction_oneliner), ("submit", Solution().reductionOperations)]:
    t0 = time.perf_counter(); r1 = fn(nums_rand); r2 = fn(nums_distinct)
    print(f"{name:8s}: {r1:>11} {r2:>11}  {time.perf_counter() - t0:.3f}s")

groups  :   785514962  1249975000  0.044s
asc     :   785514962  1249975000  0.014s
desc    :   785514962  1249975000  0.012s
count   :   785514962  1249975000  0.007s
oneliner:   785514962  1249975000  0.013s
submit  :   785514962  1249975000  0.012s


## 12. Bộ test

In [20]:
fns = [reduction_groups, reduction_rank_asc, reduction_desc, reduction_counting, reduction_oneliner,
       lambda a: Solution().reductionOperations(a)]

for nums, want in examples:
    assert reduction_simulate(nums) == want
    for fn in fns:
        assert fn(nums) == want

edge = [([1], 0), ([50000], 0), ([2, 1], 1), ([1, 2], 1), ([3, 2, 1], 3), ([1, 50000], 1),
        ([4, 4, 4, 1], 3), ([1, 4, 4, 4], 3), ([1, 2, 3, 4, 5], 10)]
for nums, want in edge:
    assert reduction_simulate(nums) == want, nums
    for fn in fns:
        assert fn(nums) == want, (fn, nums)

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 15)
    hi = random.choice([2, 5, 20, 5 * 10**4])
    nums = [random.randint(1, hi) for _ in range(n)]
    want = reduction_simulate(nums)
    for fn in fns:
        assert fn(nums) == want, (fn, nums)
print("ví dụ + edge + 1500 bộ ngẫu nhiên khớp mô phỏng trực tiếp")

ví dụ + edge + 1500 bộ ngẫu nhiên khớp mô phỏng trực tiếp


In [21]:
# tính chất: đáp án không phụ thuộc thứ tự phần tử, và = n(n-1)/2 khi mọi phần tử khác nhau
random.seed(5)
for _ in range(200):
    nums = [random.randint(1, 30) for _ in range(random.randint(1, 25))]
    sh = nums[:]; random.shuffle(sh)
    assert reduction_rank_asc(nums) == reduction_rank_asc(sh) == reduction_simulate(sh)
for n in range(1, 30):
    assert reduction_rank_asc(random.sample(range(1, 1000), n)) == n * (n - 1) // 2
print("không phụ thuộc thứ tự; mảng phân biệt cho n(n-1)/2")

không phụ thuộc thứ tự; mảng phân biệt cho n(n-1)/2


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số phần tử khác giá trị nhỏ nhất (mỗi phần tử chỉ 1 thao tác)

In [22]:
def wrong_count_not_min(nums):
    m = min(nums)
    return sum(1 for v in nums if v != m)

for nums, want in examples:
    print("sai =", wrong_count_not_min(nums), "| đúng =", want)

sai = 2 | đúng = 3
sai = 0 | đúng = 0
sai = 3 | đúng = 4


Thao tác chỉ hạ xuống **giá trị lớn kế tiếp**, không phải xuống thẳng giá trị nhỏ nhất. **Sửa:** mỗi phần tử tốn `rank(v)` thao tác.

### Lỗi 2 — Dùng hiệu giá trị (`v − min`) làm số thao tác

In [23]:
def wrong_value_diff(nums):
    m = min(nums)
    return sum(v - m for v in nums)

for nums, want in examples:
    print("sai =", wrong_value_diff(nums), "| đúng =", want)

sai = 6 | đúng = 3
sai = 0 | đúng = 0
sai = 4 | đúng = 4


Mỗi thao tác có thể hạ **nhiều đơn vị** (từ 5 xuống 3). Số thao tác phụ thuộc **số giá trị phân biệt** nằm giữa, không phụ thuộc độ lớn. **Sửa:** đếm theo hạng.

### Lỗi 3 — Sắp tăng nhưng tăng hạng ở mọi bước (quên kiểm tra `a[i] != a[i−1]`)

In [24]:
def wrong_rank_always(nums):
    a = sorted(nums); ans = 0; r = 0
    for i in range(1, len(a)):
        r += 1                                 # SAI: giá trị bằng nhau có cùng hạng
        ans += r
    return ans

for nums, want in examples:
    print("sai =", wrong_rank_always(nums), "| đúng =", want)

sai = 3 | đúng = 3
sai = 3 | đúng = 0
sai = 10 | đúng = 4


**Sửa:** chỉ tăng hạng khi gặp giá trị **mới**.

### Lỗi 4 — Sắp giảm nhưng cộng `i + 1` hoặc cộng tại mọi `i`

In [25]:
def wrong_desc_plus_one(nums):
    a = sorted(nums, reverse=True); ans = 0
    for i in range(1, len(a)):
        if a[i] != a[i - 1]:
            ans += i + 1                       # SAI: chỉ i phần tử đứng trước (chỉ số 0..i-1)
    return ans

for nums, want in examples:
    print("sai =", wrong_desc_plus_one(nums), "| đúng =", want)

sai = 5 | đúng = 3
sai = 0 | đúng = 0
sai = 6 | đúng = 4


**Sửa:** tại điểm đổi giá trị `i` có đúng `i` phần tử lớn hơn (chỉ số `0..i−1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề mô phỏng "lặp lại thao tác tới khi…" với `n` lớn ⇒ tìm **đóng góp của từng phần tử** vào tổng số thao tác.
- "Hạ xuống giá trị phân biệt kế tiếp" ⇒ đáp án theo **hạng** trong tập giá trị phân biệt (coordinate compression).
- Quy tắc phá hòa (chỉ số nhỏ nhất) thường chỉ ảnh hưởng **thứ tự**, không ảnh hưởng **số lượng** — kiểm tra bằng mô phỏng.
- Họ bài: 1338 *Reduce Array Size to The Half*, 2860, 945 *Minimum Increment to Make Array Unique*, 1846 *Maximum Element After Decreasing and Rearranging*.

## 15. Biến thể của bài

0. **Mảng đã được sắp sẵn** (tăng hoặc giảm): bỏ bước sắp ⇒ `O(n)`, bộ nhớ `O(1)`.

1. **Tăng lên** tới giá trị lớn nhất (mỗi thao tác nâng phần tử nhỏ nhất lên giá trị nhỏ kế tiếp): đối xứng — đáp án `Σ (D − 1 − rank)`.
2. **Hạ xuống thẳng** giá trị nhỏ nhất mỗi thao tác: đáp án = số phần tử khác min (Lỗi 1 trở thành đúng!).
3. **Chi phí mỗi thao tác = độ chênh** `largest − nextLargest`: tổng chi phí = `Σ (v − min)` (Lỗi 2 trở thành đúng!).
4. **Mục tiêu chỉ cần ≤ k giá trị phân biệt**: chỉ hạ các tầng trên cùng cho tới khi còn `k` tầng — đóng góp `max(0, rank − (D − k))`… cẩn thận: tầng nào bị gộp? (xem Bài 5).

In [26]:
def increase_operations(nums):
    """Biến thể 1."""
    c = Counter(nums); vals = sorted(c); D = len(vals)
    return sum((D - 1 - r) * c[v] for r, v in enumerate(vals))

def increase_simulate(nums):
    a = list(nums); ops = 0
    while True:
        smallest = min(a); bigger = [v for v in a if v > smallest]
        if not bigger: return ops
        a[a.index(smallest)] = min(bigger); ops += 1

def total_cost_diff(nums):
    """Biến thể 3: chi phí = largest - nextLargest mỗi thao tác."""
    a = list(nums); cost = 0
    while True:
        largest = max(a); smaller = [v for v in a if v < largest]
        if not smaller: return cost
        nxt = max(smaller); a[a.index(largest)] = nxt; cost += largest - nxt

random.seed(3)
for _ in range(300):
    nums = [random.randint(1, 10) for _ in range(random.randint(1, 10))]
    assert increase_operations(nums) == increase_simulate(nums)
    assert total_cost_diff(nums) == sum(v - min(nums) for v in nums)
print("biến thể 1 và 3 khớp mô phỏng")

biến thể 1 và 3 khớp mô phỏng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | đúng 3 bước của đề | `O(ans · n)` | `O(n)` | kiểm chứng |
| 1b. Theo nhóm | hạ cả tầng | `O(n + D log D)` | `O(D)` | cầu nối |
| 2. Sắp tăng + hạng | `Σ rank` | `O(n log n)` | `O(n)` | **nộp bài** |
| 3. Sắp giảm | cộng `i` tại điểm đổi | `O(n log n)` | `O(n)` | |
| 4. Counting | duyệt giá trị | `O(n + V)` | `O(V)` | giá trị nhỏ |
| 5. Counter | một dòng | `O(n + D log D)` | `O(D)` | gọn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `reductionOperations([2, 2, 2, 5])` và `reductionOperations([10, 20, 20, 30])`?

**Bài 2 (tính tay):** liệt kê từng thao tác cho `[4, 1, 4, 2]` (theo đúng quy tắc chọn chỉ số nhỏ nhất) và đếm.

**Bài 3 (điền TODO):** hoàn thiện `ops_per_element(nums)` trả về danh sách số thao tác mà **mỗi** phần tử (theo chỉ số gốc) trải qua.

**Bài 4 (tự viết lại):** viết lời giải dùng **coordinate compression** tường minh (`dict` giá trị → hạng) rồi `sum`.

**Bài 5 (mini challenge):** biến thể "dừng khi mảng còn **tối đa `k`** giá trị phân biệt" (vẫn hạ phần tử lớn nhất xuống giá trị lớn kế tiếp). Viết công thức và kiểm chứng bằng mô phỏng. (Biến thể tự đặt.)

In [27]:
def ops_per_element(nums):
    # TODO: hạng của từng phần tử trong tập giá trị phân biệt
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[2,2,2,5]`: giá trị phân biệt `2, 5` ⇒ `5` có hạng 1. (2) Đáp án `1`. (3) `[10,20,20,30]`: hạng `0, 1, 1, 2` ⇒ `4`.

**Bài 2:** (1) `largest = 4` tại chỉ số 0 ⇒ `[2,1,4,2]`. (2) `largest = 4` tại chỉ số 2 ⇒ `[2,1,2,2]`. (3) Tiếp tục hạ ba số 2 ⇒ tổng `5`.

**Bài 3:** (1) `vals = sorted(set(nums))`. (2) `rank = {v: i for i, v in enumerate(vals)}`. (3) `[rank[v] for v in nums]`.

**Bài 4:** (1) Như Bài 3. (2) `sum(...)`. (3) So với Cách 2.

**Bài 5:** (1) Chỉ các tầng trên cùng bị hạ; quá trình dừng khi số tầng còn `≤ k`. (2) Tầng hạng `r ≥ k − 1`… mọi phần tử hạng `r` hạ xuống tầng `k − 1` ⇒ đóng góp `max(0, r − (k − 1))`. (3) Kiểm chứng.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().reductionOperations([2, 2, 2, 5]) == 1 and Solution().reductionOperations([10, 20, 20, 30]) == 4

# Bài 2
trace_simulate([4, 1, 4, 2])
assert reduction_simulate([4, 1, 4, 2]) == 5

# Bài 3
def ops_per_element_ans(nums):
    vals = sorted(set(nums)); rank = {v: i for i, v in enumerate(vals)}
    return [rank[v] for v in nums]
assert ops_per_element_ans([5, 1, 3]) == [2, 0, 1]

# Bài 4
def reduction_compress(nums):
    rank = {v: i for i, v in enumerate(sorted(set(nums)))}
    return sum(rank[v] for v in nums)

random.seed(191)
for _ in range(500):
    nums = [random.randint(1, 12) for _ in range(random.randint(1, 12))]
    assert reduction_compress(nums) == reduction_simulate(nums) == sum(ops_per_element_ans(nums))
print("Bài 1-4: OK")

bắt đầu: [4, 1, 4, 2]
thao tác 1: largest=4 tại i=0, nextLargest=2 -> [2, 1, 4, 2]
thao tác 2: largest=4 tại i=2, nextLargest=2 -> [2, 1, 2, 2]
thao tác 3: largest=2 tại i=0, nextLargest=1 -> [1, 1, 2, 2]
thao tác 4: largest=2 tại i=2, nextLargest=1 -> [1, 1, 1, 2]
thao tác 5: largest=2 tại i=3, nextLargest=1 -> [1, 1, 1, 1]
-> mọi phần tử bằng 1, tổng 5 thao tác
Bài 1-4: OK


In [29]:
# Bài 5 — dừng khi còn tối đa k giá trị phân biệt
def ops_until_k_distinct(nums, k):
    rank = {v: i for i, v in enumerate(sorted(set(nums)))}
    return sum(max(0, rank[v] - (k - 1)) for v in nums)

def simulate_until_k(nums, k):
    a = list(nums); ops = 0
    while len(set(a)) > k:
        largest = max(a); nxt = max(v for v in a if v < largest)
        a[a.index(largest)] = nxt; ops += 1
    return ops

random.seed(192)
for _ in range(600):
    nums = [random.randint(1, 10) for _ in range(random.randint(1, 12))]
    k = random.randint(1, 6)
    assert ops_until_k_distinct(nums, k) == simulate_until_k(nums, k), (nums, k)
print("Bài 5: công thức max(0, rank - (k-1)) khớp mô phỏng; k=1 trùng bài gốc:",
      ops_until_k_distinct([1, 1, 2, 2, 3], 1), ops_until_k_distinct([1, 1, 2, 2, 3], 2))

Bài 5: công thức max(0, rank - (k-1)) khớp mô phỏng; k=1 trùng bài gốc: 4 1


## 20. Template nhớ nhanh + Checklist

```python
# Mỗi phần tử tốn đúng rank(v) thao tác (số giá trị phân biệt nhỏ hơn v)
a = sorted(nums); ans = r = 0
for i in range(1, len(a)):
    if a[i] != a[i - 1]:
        r += 1                 # sang giá trị phân biệt mới -> hạng tăng
    ans += r
return ans
```

**Checklist trước khi submit:**
- [ ] Thao tác hạ xuống **giá trị phân biệt kế tiếp**, không xuống thẳng min.
- [ ] Số thao tác không phụ thuộc độ lớn chênh lệch.
- [ ] Giá trị bằng nhau có cùng hạng.
- [ ] Sắp giảm: cộng `i` (không phải `i + 1`).
- [ ] Mảng một phần tử / toàn bằng nhau ⇒ `0`.